### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 5.1 (Data Generation)
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es)
### Data from FaSt-SWOT experiment (Leg 1)
# 
**DESCRIPTION**:
This script finalizes the processed datasets for publication. It applies 
a final static salinity offset (determined via CTD calibration), calculates 
standard TEOS-10 variables (Absolute Salinity, Conservative Temperature, 
Potential Density) using the GSW library, recalculates a consistent 
final conductivity, and structures the NetCDF file with official standard 
names and CF-compliant metadata.
#
INPUT: Smoothed NetCDF files (*_step4_loess.nc).
#
OUTPUT: Final NetCDF datasets (*_step5_final.nc).
### ==============================================================================

In [3]:
import xarray as xr

import numpy as np

import pandas as pd

import statsmodels.api as sm

import gsw

from pathlib import Path


# ==========================================
# CONFIGURATION (V2: filter T and C consistently, then recompute S)
# ==========================================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing")

DIR_IN = BASE_ROOT / "Data" / "Leg1" / "processed_step4_loess"
DIR_OUT = BASE_ROOT / "Data" / "Leg1" / "processed_step5_final_v2_tc_consistent"
DIR_OUT.mkdir(parents=True, exist_ok=True)

# No offset at step 5 (offset still applied in step 6)
MEDIAN_WINDOW = 5
VERTICAL_SCALE_M = 10.0
EDGE_SCANS = 8  # Preserve boundaries to avoid LOWESS edge artifacts near the surface

print("Generating step5 V2 datasets (T/C consistent filtering, WITHOUT offset)...")

def apply_loess(data, depth, scale_meters):
    mask = np.isfinite(data) & np.isfinite(depth)
    out = np.full_like(data, np.nan, dtype=float)
    if np.count_nonzero(mask) < 5:
        return out

    x = depth[mask]
    y = data[mask]
    z_range = x.max() - x.min()
    if not np.isfinite(z_range) or z_range <= 0:
        return out

    if z_range < scale_meters:
        frac = 0.8
    else:
        frac = np.clip(scale_meters / z_range, 0.02, 0.5)

    out[mask] = sm.nonparametric.lowess(y, x, frac=frac, it=0, return_sorted=False)
    return out

# ==========================================
# PROCESSING
# ==========================================
files = sorted(list(DIR_IN.glob("*_step4_loess.nc")))

for f in files:
    try:
        with xr.open_dataset(f) as ds:
            p = ds['pressure'].values
            lat, lon = ds.latitude.values, ds.longitude.values

            # Base inputs from the same processing stage (step3 variables kept inside step4 files)
            if 't1_lagged' in ds:
                t_in = ds['t1_lagged'].values
            elif 't_lagged' in ds:
                t_in = ds['t_lagged'].values
            elif 't1' in ds:
                t_in = ds['t1'].values
            else:
                raise KeyError('No temperature variable found (t1_lagged/t_lagged/t1)')

            c_in = ds['c1'].values

            # 1) Consistent filtering on T and C (median + LOESS both)
            t_med = pd.Series(t_in).rolling(window=MEDIAN_WINDOW, center=True, min_periods=1).median().values
            c_med = pd.Series(c_in).rolling(window=MEDIAN_WINDOW, center=True, min_periods=1).median().values

            t_consistent = apply_loess(t_med, p, scale_meters=VERTICAL_SCALE_M)
            c_consistent = apply_loess(c_med, p, scale_meters=VERTICAL_SCALE_M)

            # Replace edges with median-filtered values to suppress boundary artifacts
            n_edge = min(EDGE_SCANS, len(p))
            if n_edge > 0:
                t_consistent[:n_edge] = t_med[:n_edge]
                t_consistent[-n_edge:] = t_med[-n_edge:]
                c_consistent[:n_edge] = c_med[:n_edge]
                c_consistent[-n_edge:] = c_med[-n_edge:]

            # 2) Recompute practical salinity from filtered T and C
            s_final = gsw.SP_from_C(c_consistent, t_consistent, p)

            # 3) TEOS-10 derived variables
            SA = gsw.SA_from_SP(s_final, p, lon, lat)
            CT = gsw.CT_from_t(SA, t_consistent, p)
            sigma0 = gsw.sigma0(SA, CT)

            # 4) Final conductivity consistent with final T,S
            c_final = gsw.C_from_SP(s_final, t_consistent, p)

            # 5) Final dataset
            ds_final = ds.copy(deep=True)
            ds_final['temperature_conservative'] = (('scan',), CT)
            ds_final['salinity_practical'] = (('scan',), s_final)
            ds_final['conductivity_final'] = (('scan',), c_final)
            ds_final['sigma_theta'] = (('scan',), sigma0)

            # Keep explicit V2 diagnostics
            ds_final['t1_filtered_consistent'] = (('scan',), t_consistent)
            ds_final['c1_filtered_consistent'] = (('scan',), c_consistent)

            ds_final['temperature_conservative'].attrs = {'units': 'degC', 'standard_name': 'sea_water_conservative_temperature'}
            ds_final['salinity_practical'].attrs = {
                'units': 'PSU',
                'standard_name': 'sea_water_practical_salinity',
                'comment': 'V2: computed from consistently filtered T and C (median+LOESS), no offset in step5'
            }
            ds_final['conductivity_final'].attrs = {'units': 'mS/cm', 'standard_name': 'sea_water_electrical_conductivity'}
            ds_final['sigma_theta'].attrs = {'units': 'kg/m^3', 'standard_name': 'sea_water_sigma_theta'}
            ds_final['t1_filtered_consistent'].attrs = {'units': 'degC', 'comment': 'V2 filtered temperature (median+LOESS, edge-preserved)'}
            ds_final['c1_filtered_consistent'].attrs = {'units': 'mS/cm', 'comment': 'V2 filtered conductivity (median+LOESS, edge-preserved)'}

            ds_final.attrs['step5_version'] = 'v2_tc_consistent_filtering'
            ds_final.attrs['step5_notes'] = 'T and C filtered consistently before salinity recomputation; LOWESS edge-preserving boundaries'

            out_name = f.name.replace('_step4_loess.nc', '_step5_final_v2.nc')
            ds_final.to_netcdf(DIR_OUT / out_name)
            print(f"   ✅ V2 dataset created: {out_name}")

    except Exception as e:
        print(f"   ❌ Error in {f.name}: {e}")

print(f"\nProcess completed. {len(files)} files saved in {DIR_OUT}")

Generating step5 V2 datasets (T/C consistent filtering, WITHOUT offset)...
   ✅ V2 dataset created: imedea-socib_fast-swot1_mvp_0009_step5_final_v2.nc
   ✅ V2 dataset created: imedea-socib_fast-swot1_mvp_0011_step5_final_v2.nc
   ✅ V2 dataset created: imedea-socib_fast-swot1_mvp_0013_step5_final_v2.nc
   ✅ V2 dataset created: imedea-socib_fast-swot1_mvp_0015_step5_final_v2.nc
   ✅ V2 dataset created: imedea-socib_fast-swot1_mvp_0019_step5_final_v2.nc
   ✅ V2 dataset created: imedea-socib_fast-swot1_mvp_0021_step5_final_v2.nc
   ✅ V2 dataset created: imedea-socib_fast-swot1_mvp_0023_step5_final_v2.nc
   ✅ V2 dataset created: imedea-socib_fast-swot1_mvp_0025_step5_final_v2.nc
   ✅ V2 dataset created: imedea-socib_fast-swot1_mvp_0026_step5_final_v2.nc
   ✅ V2 dataset created: imedea-socib_fast-swot1_mvp_0028_step5_final_v2.nc
   ✅ V2 dataset created: imedea-socib_fast-swot1_mvp_0033_step5_final_v2.nc
   ✅ V2 dataset created: imedea-socib_fast-swot1_mvp_0037_step5_final_v2.nc
   ✅ V2 datas

### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 5.2 (Validation Figures)
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es) 
### Data from FaSt-SWOT experiment (Leg 1)
# 
**DESCRIPTION**:
This script generates the final publication-ready figures comparing the 
fully calibrated MVP profiles against their nearest CTD casts. The 3-panel 
plots display Temperature, Conductivity, and Salinity, illustrating the raw 
MVP data, the final processed MVP data, and the CTD. A text box reports the 
percentage of error reduction achieved by the pipeline.
#
INPUT: Raw MVP files (Step 1), Final MVP files (Step 5), and Reference CTD (.cnv).
#
OUTPUT: Final comparison plots (.png) in 'FIGURES_PAPER_FINAL_RESULTS'.
### ==============================================================================

In [4]:
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from geopy.distance import geodesic
import gsw
import re
import io
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURATION (V2 outputs)
# ==========================================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing")

DIR_MVP_RAW = BASE_ROOT / "Data" / "Leg1" / "processed_step1_highres_qc"
DIR_MVP_FINAL = BASE_ROOT / "Data" / "Leg1" / "processed_step5_final_v2_tc_consistent"
DIR_CTD = Path(r"C:/Users/ASUS/OneDrive - Universitat de les Illes Balears/SWOT/CTD/CTD_data/leg1/HM/")

OUTDIR = BASE_ROOT / "Figures" / "FIGURES_PAPER_FINAL_RESULTS_V2_TC_CONSISTENT"
OUTDIR.mkdir(parents=True, exist_ok=True)

# Parameters
MAX_DIST_KM = 0.5
MAX_TIME_MIN = 60.0
Z_MAX_PLOT = 100.0
CTD_SMOOTH_WINDOW = 8

# ==========================================
# 2. FUNCTIONS
# ==========================================

def read_ctd_robust(path):
    try:
        with open(path, 'r', encoding='latin-1') as f:
            lines = f.readlines()
        lat, lon, time_val, start_idx = np.nan, np.nan, pd.NaT, 0
        col_map = {}
        for i, line in enumerate(lines):
            if 'NMEA Latitude' in line:
                p = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(p) >= 2:
                    lat = float(p[0]) + float(p[1]) / 60 * (-1 if 'S' in line else 1)
            if 'NMEA Longitude' in line:
                p = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(p) >= 2:
                    lon = float(p[0]) + float(p[1]) / 60 * (-1 if 'W' in line else 1)
            if 'start_time' in line:
                try:
                    time_val = pd.to_datetime(line.split('=')[1].strip().split('[')[0])
                except Exception:
                    pass
            if '# name' in line:
                match = re.search(r"name (\d+)", line)
                if match:
                    idx = int(match.group(1))
                    name = line.split('=')[1].split(':')[0].strip().lower()
                    if name in ['prdm', 'pres', 'pressure']:
                        col_map['p'] = idx
                    elif name in ['t090c', 't190c', 'temp', 'tv290c']:
                        col_map['t'] = idx
                    elif name in ['sal00', 'sal11', 'sal', 'psal']:
                        col_map['s'] = idx
                    elif name in ['c0s/m', 'c1s/m', 'cond', 'c0ms/cm']:
                        col_map['c'] = idx
            if '*END*' in line:
                start_idx = i + 1
                break

        df = pd.read_csv(io.StringIO(''.join(lines[start_idx:])), delim_whitespace=True, header=None)
        p, t, s = df.iloc[:, col_map['p']].values, df.iloc[:, col_map['t']].values, df.iloc[:, col_map['s']].values
        c = df.iloc[:, col_map['c']].values if 'c' in col_map else None
        if c is not None and np.nanmean(c) < 10:
            c *= 10.0
        return {'lat': lat, 'lon': lon, 'time': time_val, 'id': path.stem, 'p': p, 't': t, 's': s, 'c': c}
    except Exception:
        return None

def load_mvp_catalog(mvp_dir):
    data = []
    files = sorted(list(mvp_dir.glob("*.nc")))
    for f in files:
        try:
            with xr.open_dataset(f) as ds:
                lat = ds.latitude.values.flatten()[0] if 'latitude' in ds else np.nan
                lon = ds.longitude.values.flatten()[0] if 'longitude' in ds else np.nan
                t_val = pd.to_datetime(ds.time.values.flatten()[0]) if 'time' in ds else pd.NaT
                if np.isfinite(lat):
                    data.append({'file': f.name, 'lat': lat, 'lon': lon, 'time': t_val, 'path': f})
        except Exception:
            pass
    return pd.DataFrame(data)

# ==========================================
# 3. PROCESSING AND PLOTTING
# ==========================================

print("Generating comparative figures with V2 step5 data...")
df_mvp = load_mvp_catalog(DIR_MVP_RAW)

if df_mvp.empty:
    print("❌ Error: No MVP data available.")
else:
    for ctd_p in sorted(DIR_CTD.glob("d*.cnv")):
        ctd = read_ctd_robust(ctd_p)
        if not ctd or pd.isna(ctd['time']):
            continue

        df_mvp['dist_km'] = [geodesic((ctd['lat'], ctd['lon']), (m.lat, m.lon)).km for m in df_mvp.itertuples()]
        df_mvp['dt_min'] = [(abs(m.time - ctd['time']).total_seconds() / 60.0) for m in df_mvp.itertuples()]
        matches = df_mvp[(df_mvp['dist_km'] <= MAX_DIST_KM) & (df_mvp['dt_min'] <= MAX_TIME_MIN)].copy()

        if matches.empty:
            continue

        best_match = matches.loc[matches['dist_km'].idxmin()]
        raw_path = DIR_MVP_RAW / best_match['file']
        final_path = DIR_MVP_FINAL / best_match['file'].replace("_step1_qc.nc", "_step5_final_v2.nc")

        if not final_path.exists():
            continue

        try:
            with xr.open_dataset(final_path) as ds_f, xr.open_dataset(raw_path) as ds_r:
                p_p = ds_f.pressure.values
                t_p = ds_f.temperature_conservative.values
                s_p = ds_f.salinity_practical.values
                c_p = ds_f.conductivity_final.values

                p_r, t_r, c_r = ds_r.pressure.values, ds_r.t1.values, ds_r.c1.values
                if np.nanmean(c_r) < 10:
                    c_r *= 10
                s_r = gsw.SP_from_C(c_r, t_r, p_r)

            p_grid = np.arange(10, Z_MAX_PLOT, 1.0)
            s_ctd_smooth = pd.Series(ctd['s']).rolling(window=CTD_SMOOTH_WINDOW, center=True, min_periods=1).mean().values
            s_ctd_i = np.interp(p_grid, ctd['p'], s_ctd_smooth)
            s_raw_i = np.interp(p_grid, p_r, s_r)
            s_fin_i = np.interp(p_grid, p_p, s_p)

            mask = np.isfinite(s_ctd_i) & np.isfinite(s_raw_i) & np.isfinite(s_fin_i)
            rmsd_raw = np.sqrt(np.mean((s_ctd_i[mask] - s_raw_i[mask]) ** 2))
            rmsd_fin = np.sqrt(np.mean((s_ctd_i[mask] - s_fin_i[mask]) ** 2))
            imp = (1 - rmsd_fin / rmsd_raw) * 100

            fig, axs = plt.subplots(1, 3, figsize=(16, 7), sharey=True, constrained_layout=True)
            kw_raw = {'s': 5, 'color': '#CCCCCC', 'alpha': 0.7, 'label': 'MVP Raw'}
            kw_fin = {'s': 4, 'color': 'red', 'alpha': 0.8, 'label': 'MVP Processed V2'}

            axs[0].scatter(t_r, p_r, **kw_raw)
            axs[0].scatter(t_p, p_p, **kw_fin)
            t_ctd_smooth = pd.Series(ctd['t']).rolling(window=CTD_SMOOTH_WINDOW, center=True, min_periods=1).mean().values
            axs[0].plot(t_ctd_smooth, ctd['p'], color='black', lw=2, label='CTD Reference')
            axs[0].set_xlabel("Temperature (°C)", fontsize=14)
            axs[0].set_ylabel("Pressure (dbar)", fontsize=14)
            axs[0].tick_params(axis='both', which='both', labelsize=14)

            axs[1].scatter(c_r, p_r, **kw_raw)
            axs[1].scatter(c_p, p_p, **kw_fin)
            if ctd['c'] is not None:
                c_ctd_smooth = pd.Series(ctd['c']).rolling(window=CTD_SMOOTH_WINDOW, center=True, min_periods=1).mean().values
                axs[1].plot(c_ctd_smooth, ctd['p'], color='black', lw=2)
            axs[1].set_xlabel("Conductivity (mS/cm)", fontsize=14)
            axs[1].tick_params(axis='both', which='both', labelsize=14)

            axs[2].scatter(s_r, p_r, **kw_raw)
            axs[2].scatter(s_p, p_p, **kw_fin)
            axs[2].plot(s_ctd_smooth, ctd['p'], color='black', lw=2)
            axs[2].set_xlabel("Salinity", fontsize=14)
            axs[2].tick_params(axis='both', which='both', labelsize=14)

            for ax in axs:
                ax.set_ylim(Z_MAX_PLOT, 0)
                ax.grid(True, alpha=0.3)

            axs[0].legend(loc='best', fontsize=11)
            fig.suptitle(f"V2 Comparison | {best_match['file']} | RMSD reduction: {imp:.1f}%", fontsize=13)

            out_name = f"V2_{best_match['file'].replace('.nc', '')}_comparison.png"
            fig.savefig(OUTDIR / out_name, dpi=150)
            plt.close(fig)
            print(f"✅ Figure created: {out_name}")

        except Exception as e:
            print(f"❌ Error plotting {best_match['file']}: {e}")

print(f"\nDone. V2 figures saved in: {OUTDIR}")

Generating comparative figures with V2 step5 data...
✅ Figure created: V2_imedea-socib_fast-swot1_mvp_0153_step1_qc_comparison.png
✅ Figure created: V2_imedea-socib_fast-swot1_mvp_0202_step1_qc_comparison.png
✅ Figure created: V2_imedea-socib_fast-swot1_mvp_0617_step1_qc_comparison.png
✅ Figure created: V2_imedea-socib_fast-swot1_mvp_0627_step1_qc_comparison.png
✅ Figure created: V2_imedea-socib_fast-swot1_mvp_0632_step1_qc_comparison.png

Done. V2 figures saved in: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing\Figures\FIGURES_PAPER_FINAL_RESULTS_V2_TC_CONSISTENT
